# Intro

This notebook works with the GPS data related to a video recording of a camera mounted on a bike. 

This notebook assumes that there is already a file route.gpx extracted from a video (see readme.md for instructions using exiftool)


# This notebook
In this notebook:
- Read GPX, visualize on map
- Determine speed/acceleration based on GPS data
- Use GPS location and speed to select movie segments

# Read gpx


In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import fiona 

from moviepy.editor import VideoFileClip
from IPython.display import display, Video, clear_output
import ipywidgets as widgets
import os

from gps_basics import parse_gpx, get_coordinates_and_times, create_gps_map
from gps_analysis import compute_acceleration, compute_speeds, plot_speed
from helper_functions import get_intervals_from_array, intervals_to_time_segments

In [ ]:
# Load gpx, extract data (coordinates and time in video)
gpx_file = "route.gpx"

# Print available layers in the GPX file
layers = fiona.listlayers(gpx_file)
print("Available layers:", layers)

# Load GPX data 
gpx = parse_gpx(gpx_file)

# Extract coordinates and times 
coordinates, times, times_string = get_coordinates_and_times(gpx)

# Basic map of GPS data with time of video in tooltip

In [ ]:
# By providing the times to tooltip_list, the time is displayed in the tooltips
create_gps_map(coordinates, zoom_start=14, tooltip_list = times_string)

## Compute speed based on GPX data

In [4]:
speeds, smoothed_speeds = compute_speeds(coordinates, times, smoothing_method='moving_average', window_size=7)

speeds_kmh = [s * 3.6 for s in speeds]
smoothed_speeds_kmh = [s * 3.6 for s in smoothed_speeds]

# best to compute acceleration of smoothed speeds
accelerations = compute_acceleration(smoothed_speeds, times)

In [ ]:
plot_speed(times_string, smoothed_speeds_kmh, in_kmh = True)

plot_speed(times_string, smoothed_speeds, accelerations)

In [ ]:
# color the gpx map by speed (provide the smoothed speeds to color_list)
create_gps_map(coordinates, zoom_start=14, 
               tooltip_list = times_string, 
               color_list = smoothed_speeds_kmh,
               color_list_name = "Speed (km/h)")

## Find segments in video when speed is below 5 km/h

In [ ]:
# Get indices where smoothed_speeds is less than 5
# First, convert them to contiguous intervals, then to time segments
low_speed_indices = np.where(np.array(smoothed_speeds_kmh) < 5)[0]

intervals = get_intervals_from_array(low_speed_indices)
time_segments = intervals_to_time_segments(intervals, times, to_minutes=False)
print(time_segments)

### Example: show the video segments found above 

In [ ]:
# Load the movie once
movie = VideoFileClip("MOVIE.MP4")

# Pre-extract the clips into memory
clips = [movie.subclip(start, end) for start, end in intervals]

# Create widgets
index = widgets.IntText(value=0, description="Clip #")
prev_btn = widgets.Button(description="Previous")
next_btn = widgets.Button(description="Next")

output = widgets.Output()

def show_clip(i):
    """Display the selected clip inline."""
    clear_output(wait=True)
    i = int(i)
    if 0 <= i < len(clips):
        temp_name = f"clip_{i}.mp4"
        if os.path.exists(temp_name):
            display(Video(temp_name, embed=True))
        else:
            clips[i].write_videofile(temp_name, codec="libx264", audio_codec="aac", verbose=False, logger=None)
            display(Video(temp_name, embed=True))
        #clips[i].write_videofile(temp_name, codec="libx264", audio_codec="aac", verbose=False, logger=None)
        #display(Video(temp_name, embed=True))
        display(widgets.HBox([prev_btn, next_btn, index]))
    else:
        print("Invalid clip index")

def on_prev(b):
    index.value = (index.value - 1) % len(clips)
    show_clip(index.value)

def on_next(b):
    index.value = (index.value + 1) % len(clips)
    show_clip(index.value)

# Bind events
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)

# Display the first clip
show_clip(0)

In [ ]:
import os

remove = False # Set to True to remove temporary files after viewing

if remove:
    # Remove temporary clip files
    for i in range(len(clips)):
        temp_name = f"clip_{i}.mp4"
        if os.path.exists(temp_name):
            os.remove(temp_name)
            print(f"Removed {temp_name}")